In [1]:
#Import tools for text preprocessing, data manipulation and visualization.
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt

#natural language processing toolkit
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
stop_words -= {'not', 'never', 'no', 'nor', 'none'}  #stop words are commonly used words that are often removed from text data during preprocessing to improve the performance of natural language processing models.
lemmatizer = WordNetLemmatizer()#Lemmatization is the process of reducing words to their base or root form, known as a lemma. This helps in normalizing the text data and improving the performance of natural language processing models.

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\visha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\visha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\visha\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Task 2: Text Processing and Normalisation

Text preprocessing transforms raw, noisy review text into a clean, structured format that machine learning models can learn from effectively. Each step below addresses a specific type of noise or ambiguity present in user-generated content.

---

### 2.1 Library Setup and NLTK Downloads

The preprocessing pipeline relies on four core NLTK resources:

| Resource | Purpose |
|---|---|
| `punkt` | Tokenisation — splits text into individual word tokens |
| `stopwords` | Common word list for filtering low-information words |
| `wordnet` | Lexical database used by the lemmatizer to find base forms |

---

In [2]:
#used https://www.kaggle.com/code/nmaguette/up-to-date-list-of-slangs-for-text-preprocessing - a notebook with a comprehensive list of slang and abbreviations to expand in the preprocessing pipeline.More abbreviations were added to the list as well.
abbreviations = {
    "$" : " dollar ",
    "€" : " euro ",
    "4ao" : "for adults only",
    "a.m" : "before midday",
    "a3" : "anytime anywhere anyplace",
    "aamof" : "as a matter of fact",
    "acct" : "account",
    "adih" : "another day in hell",
    "afaic" : "as far as i am concerned",
    "afaict" : "as far as i can tell",
    "afaik" : "as far as i know",
    "afair" : "as far as i remember",
    "afk" : "away from keyboard",
    "app" : "application",
    "approx" : "approximately",
    "apps" : "applications",
    "asap" : "as soon as possible",
    "asl" : "age, sex, location",
    "atk" : "at the keyboard",
    "ave." : "avenue",
    "aymm" : "are you my mother",
    "ayor" : "at your own risk", 
    "b&b" : "bed and breakfast",
    "b+b" : "bed and breakfast",
    "b.c" : "before christ",
    "b2b" : "business to business",
    "b2c" : "business to customer",
    "b4" : "before",
    "b4n" : "bye for now",
    "b@u" : "back at you",
    "bae" : "before anyone else",
    "bak" : "back at keyboard",
    "bbbg" : "bye bye be good",
    "bbc" : "british broadcasting corporation",
    "bbias" : "be back in a second",
    "bbl" : "be back later",
    "bbs" : "be back soon",
    "be4" : "before",
    "bfn" : "bye for now",
    "blvd" : "boulevard",
    "bout" : "about",
    "brb" : "be right back",
    "bros" : "brothers",
    "brt" : "be right there",
    "bsaaw" : "big smile and a wink",
    "btw" : "by the way",
    "bwl" : "bursting with laughter",
    "c/o" : "care of",
    "cet" : "central european time",
    "cf" : "compare",
    "clg": "college",
    "cm" : "centimeter",
    "cia" : "central intelligence agency",
    "csl" : "can not stop laughing",
    "cu" : "see you",
    "cul8r" : "see you later",
    "cv" : "curriculum vitae",
    "cwot" : "complete waste of time",
    "cya" : "see you",
    "cyt" : "see you tomorrow",
    "dae" : "does anyone else",
    "dbmib" : "do not bother me i am busy",
    "diy" : "do it yourself",
    "dm" : "direct message",
    "dwh" : "during work hours",
    "e123" : "easy as one two three",
    "eet" : "eastern european time",
    "eg" : "example",
    "embm" : "early morning business meeting",
    "encl" : "enclosed",
    "encl." : "enclosed",
    "etc" : "and so on",
    "faq" : "frequently asked questions",
    "fawc" : "for anyone who cares",
    "fb" : "facebook",
    "fc" : "fingers crossed",
    "fig" : "figure",
    "fimh" : "forever in my heart", 
    "ft." : "feet",
    "ft" : "featuring",
    "ftl" : "for the loss",
    "ftw" : "for the win",
    "fwiw" : "for what it is worth",
    "fyi" : "for your information",
    "g9" : "genius",
    "gahoy" : "get a hold of yourself",
    "gal" : "get a life",
    "gcse" : "general certificate of secondary education",
    "gfn" : "gone for now",
    "gg" : "good game",
    "gl" : "good luck",
    "glhf" : "good luck have fun",
    "gmt" : "greenwich mean time",
    "gmta" : "great minds think alike",
    "gn" : "good night",
    "g.o.a.t" : "greatest of all time",
    "goat" : "greatest of all time",
    "goi" : "get over it",
    "gps" : "global positioning system",
    "gr8" : "great",
    "gratz" : "congratulations",
    "gyal" : "girl",
    "h&c" : "hot and cold",
    "hp" : "horsepower",
    "hr" : "hour",
    "hlo" : "hello",
    "hrh" : "his royal highness",
    "ht" : "height",
    "ibrb" : "i will be right back",
    "ic" : "i see",
    "icq" : "i seek you",
    "icymi" : "in case you missed it",
    "idc" : "i do not care",
    "idgadf" : "i do not give a damn fuck",
    "idgaf" : "i do not give a fuck",
    "idk" : "i do not know",
    "ie" : "that is",
    "i.e" : "that is",
    "ifyp" : "i feel your pain",
    "IG" : "instagram",
    "iirc" : "if i remember correctly",
    "ilu" : "i love you",
    "ily" : "i love you",
    "imho" : "in my humble opinion",
    "imo" : "in my opinion",
    "imu" : "i miss you",
    "iow" : "in other words",
    "irl" : "in real life",
    "j4f" : "just for fun",
    "jic" : "just in case",
    "jk" : "just kidding",
    "jsyk" : "just so you know",
    "l8r" : "later",
    "lb" : "pound",
    "lbs" : "pounds",
    "ldr" : "long distance relationship",
    "lmao" : "laugh my ass off",
    "lmfao" : "laugh my fucking ass off",
    "lol" : "laughing out loud",
    "ltd" : "limited",
    "ltns" : "long time no see",
    "m8" : "mate",
    "mf" : "motherfucker",
    "mfs" : "motherfuckers",
    "ml": "machine learning",
    "mfw" : "my face when",
    "mofo" : "motherfucker",
    "mph" : "miles per hour",
    "mr" : "mister",
    "mrw" : "my reaction when",
    "ms" : "miss",
    "mte" : "my thoughts exactly",
    "nagi" : "not a good idea",
    "nbc" : "national broadcasting company",
    "nbd" : "not big deal",
    "nfs" : "not for sale",
    "ngl" : "not going to lie",
    "nhs" : "national health service",
    "nrn" : "no reply necessary",
    "nsfl" : "not safe for life",
    "nsfw" : "not safe for work",
    "nth" : "nice to have",
    "nvr" : "never",
    "nyc" : "new york city",
    "oc" : "original content",
    "og" : "original",
    "ohp" : "overhead projector",
    "oic" : "oh i see",
    "omdb" : "over my dead body",
    "omg" : "oh my god",
    "omw" : "on my way",
    "pl": "please",
    "pls" : "please",
    "plz" : "please",
    "p.a" : "per annum",
    "p.m" : "after midday",
    "pm" : "prime minister",
    "poc" : "people of color",
    "pov" : "point of view",
    "pp" : "pages",
    "ppl" : "people",
    "prw" : "parents are watching",
    "ps" : "postscript",
    "pt" : "point",
    "ptb" : "please text back",
    "pto" : "please turn over",
    "qpsa" : "what happens",
    "ratchet" : "rude",
    "rbtl" : "read between the lines",
    "rlrt" : "real life retweet", 
    "rofl" : "rolling on the floor laughing",
    "roflol" : "rolling on the floor laughing out loud",
    "rotflmao" : "rolling on the floor laughing my ass off",
    "rt" : "retweet",
    "ruok" : "are you ok",
    "sfw" : "safe for work",
    "sk8" : "skate",
    "smh" : "shake my head",
    "sq" : "square",
    "srsly" : "seriously", 
    "ssdd" : "same stuff different day",
    "tbh" : "to be honest",
    "tbs" : "tablespooful",
    "tbsp" : "tablespooful",
    "tfw" : "that feeling when",
    "thks" : "thank you",
    "tho" : "though",
    "thx" : "thank you",
    "tia" : "thanks in advance",
    "til" : "today i learned",
    "tl;dr" : "too long i did not read",
    "tldr" : "too long i did not read",
    "tmb" : "tweet me back",
    "tntl" : "trying not to laugh",
    "ttyl" : "talk to you later",
    "u" : "you",
    "u2" : "you too",
    "u4e" : "yours for ever",
    "utc" : "coordinated universal time",
    "w/" : "with",
    "w/o" : "without",
    "w8" : "wait",
    "wassup" : "what is up",
    "wb" : "welcome back",
    "wtf" : "what the fuck",
    "wtg" : "way to go",
    "wtpa" : "where the party at",
    "wuf" : "where are you from",
    "wuzup" : "what is up",
    "wywh" : "wish you were here",
    "yd" : "yard",
    "ygtr" : "you got that right",
    "ynk" : "you never know",
    "zzz" : "sleeping bored and tired",
    "r": "are",
    "y": "why",
    "n": "and",
    "ur": "your",
    "gonna": "going to",
    "wanna": "want to",
    "gotta": "got to",
    "kinda": "kind of",
    "sorta": "sort of",
}

def expand_abbreviations(text): #expand abbreviations in the text using the abbreviations dictionary
    for abbr in ["$", "€", "b&b", "b+b", "c/o", "h&c", "w/", "w/o", "b@u"]:
        if abbr in abbreviations and abbr in text:
            text = text.replace(abbr, abbreviations[abbr])
    
    # Split text into words to handle other abbreviations
    words = text.split()
    expanded = []# Create a new list to hold expanded words
    
    for word in words:
        clean_word = word.lower().strip('.,!?;:\'"') #Remove punctuation for matching
        if clean_word in abbreviations:# If the cleaned word is in the abbreviations dictionary, append the expanded form
            expanded.append(abbreviations[clean_word])
        else:
            expanded.append(word)# If not, append the original word
    return ' '.join(expanded)# Join the expanded words back into a single string


def remove_emojis(text):
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002702-\U000027B0"  # dingbats
        u"\U000024C2-\U0001F251"  # enclosed characters
        u"\U0001F900-\U0001F9FF"  # supplemental symbols
        u"\U0001FA70-\U0001FAFF"  # extended symbols
        "]+", 
        flags=re.UNICODE
    )
    return emoji_pattern.sub(r'', text)

def handle_negations(text):
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"won't", "will not", text)
    text = re.sub(r"can't", "cannot", text)
    return text

def preprocess(text):
    # Remove emojis first
    text = remove_emojis(text)
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Normalization (lowercase)
    text = text.lower()

    # Expand abbreviations
    text = expand_abbreviations(text)
    
    #  Handle negations before removing punctuation to preserve the meaning of negated words
    text = handle_negations(text)
    
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    
    # Tokenization
    tokens = word_tokenize(text)
    
    # Remove stopwords
    tokens = [w for w in tokens if w not in stop_words]
    
    # Lemmatization
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    
    return " ".join(tokens)

test_text = "omg lol brb! this is gr8 😂 btw idk wtf is happening"
print("Original:", test_text)
print("Processed:", preprocess(test_text))

Original: omg lol brb! this is gr8 😂 btw idk wtf is happening
Processed: oh god laughing loud right back great way not know fuck happening


---

### 2.2 Stop Word Handling 
Standard NLTK stop words are loaded, but **five negation words are deliberately retained**:

```python
stop_words -= {'not', 'never', 'no', 'nor', 'none'}
```

**Justification:** Removing negations is one of the most damaging mistakes in sentiment analysis preprocessing. The difference between *"good service"* and *"not good service"* is entirely carried by the word *"not"* — stripping it would flip the sentiment of the review. Retaining these words preserves the polarity of negated phrases, which is especially important given that 50.3% of this dataset is negative.

**Trade-off:** Keeping negations slightly increases vocabulary size and may introduce noise in ambiguous constructions like *"not bad"* (which is weakly positive), but this cost is far outweighed by the benefit of preserving explicit negative signals.

---

### 2.3 Abbreviation Expansion

A comprehensive dictionary of ~200 internet slang terms and abbreviations is defined and applied before any other text transformation.

**Justification:** User-generated reviews are rich in informal language — *"omg lol brb gr8"* is meaningless to a TF-IDF vectoriser or a model trained on standard English vocabulary. Expanding these terms recovers their semantic meaning:

```
omg lol brb gr8  →  oh god laughing loud right back great
```

**Design choices worth noting:**
- Special-character abbreviations (`$`, `€`, `w/`, `b&b`) are handled **first**, before the text is split into words, because they would be stripped by later punctuation removal
- Word matching uses `strip('.,!?;:\'"')` to handle abbreviations that appear mid-sentence with trailing punctuation (e.g. `"brb!"` → matched as `"brb"`)

**Trade-off:** The dictionary approach is brittle — it only catches known abbreviations and may incorrectly expand legitimate words that happen to match an abbreviation key (e.g. `"r"` → `"are"`, which could misfire on names or codes). A learned subword tokeniser (e.g. BPE) would handle unseen slang more robustly, but adds significant complexity.

---

### 2.4 Emoji Removal

A Unicode regex pattern strips all emoji characters across six Unicode blocks (emoticons, symbols, transport, flags, supplemental, and extended symbols).

**Justification:** Emojis cannot be processed by standard NLP pipelines and would either be dropped silently or cause tokenisation errors. Removing them cleanly prevents downstream issues.

**Trade-off:** Emojis do carry sentiment signal — 😂 and 😡 communicate very different emotions. A more sophisticated approach would *replace* emojis with their text description (e.g. 😂 → `"face tears joy"`) before removal. For this pipeline, removal is the simpler and safer choice.

---

### 2.5 Negation Handling

Contractions are expanded before punctuation is removed:

```python
n't  →  not
won't  →  will not
can't  →  cannot
```

**Justification:** If punctuation is removed first, `"don't"` becomes `"dont"` — an out-of-vocabulary token that is then filtered as a stop word, silently deleting the negation. Expanding contractions *before* punctuation removal ensures *"not"* survives into the final token set.

**Order dependency:** This step must come **before** punctuation removal and **after** lowercasing. The ordering in the `preprocess()` function is deliberate and correct.

---

### 2.6 The Full Preprocessing Pipeline

The `preprocess()` function applies all steps in a fixed, justified order:

| Step | Method | Why this order |
|---|---|---|
| 1. Emoji removal | Unicode regex | Must precede lowercasing to match Unicode ranges correctly |
| 2. HTML tag removal | `<.*?>` regex | Removes structural noise before any text analysis |
| 3. URL removal | `https?://` regex | URLs are uninformative and would fragment badly during tokenisation |
| 4. Lowercasing | `.lower()` | Normalises case before abbreviation matching |
| 5. Abbreviation expansion | Dictionary lookup | Must occur before punctuation removal |
| 6. Negation handling | Regex contraction expansion | Must occur before punctuation removal |
| 7. Punctuation removal | `[^\w\s]` regex | Applied after all text-level transformations are complete |
| 8. Tokenisation | `word_tokenize()` | NLTK's tokeniser handles edge cases better than `.split()` |
| 9. Stop word removal | Set lookup | Applied to tokens, not raw text |
| 10. Lemmatisation | `WordNetLemmatizer` | Final step — operates on clean, filtered tokens |

---

In [3]:
#Load the training and testing data from CSV files, which contain the reviews and their corresponding ratings.
df_train_data = pd.read_csv('data/train_english.csv')
df_test_data = pd.read_csv('data/test_english.csv')

# Apply preprocessing
df_train_data['processed'] = df_train_data['text'].apply(preprocess)
df_test_data['processed'] = df_test_data['text'].apply(preprocess)

print(df_train_data[['text', 'processed']].head()) #Print original and processed text for the first 5 rows of the training data to verify preprocessing


X_train = df_train_data['processed'] #Use the processed text for training
y_train = df_train_data['rating'] #Use the original 'rating' column for training labels, not the processed text

X_test = df_test_data['processed'] #Use the processed text for testing
y_test = df_test_data['rating']#Use the original 'rating' column for testing labels, not the processed text

print("\nData shapes:") #Print the shapes of the training and testing data to verify that they are correctly prepared for model training and evaluation.
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")

KeyboardInterrupt: 

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Find a review with rich vocabulary — longer reviews work best
rich_review = df_train_data[df_train_data['text'].apply(
    lambda x: len(str(x).split()) > 20)]['text'].iloc[0]

print("Original Review:\n", rich_review, "\n")

tokens = word_tokenize(rich_review.lower())
tokens = [w for w in tokens if w.isalpha()]  # remove punctuation tokens

import pandas as pd
comparison = pd.DataFrame({
    'Original':   tokens,
    'Stemmed':    [stemmer.stem(w) for w in tokens],
    'Lemmatized': [lemmatizer.lemmatize(w) for w in tokens]
})

# Highlight only rows where stemming and lemmatization differ
differs = comparison[comparison['Stemmed'] != comparison['Lemmatized']]
print("Tokens where stemming and lemmatization differ:")
print(differs.to_string(index=False))

Original Review:
 This place is TERRIBLE; the people in charge are the worst part, by far.  Yeah, the place is beautiful inside but the people running it are not. The director can be nice at first but she can truly careless about you or your family.
Honestly, I only gave it two star because of some of the dedicated employees. I really enjoy the dedication they bring to the table. They are the reasons you should want to bring your children. However, I suggest,for the place to become a better experience, they should  re-evaluate the director and assistant director. 

Tokens where stemming and lemmatization differ:
  Original  Stemmed Lemmatized
      this      thi       this
  terrible  terribl   terrible
    people    peopl     people
    charge    charg     charge
 beautiful   beauti  beautiful
    inside    insid     inside
    people    peopl     people
   running      run    running
     truly    truli      truly
    family   famili     family
  honestly honestli   honestly
      on

### 2.7 Stemming vs. Lemmatisation — Justification for Choosing Lemmatisation

A side-by-side comparison is run on a real review to evaluate both approaches. The results reveal a clear pattern:

| Original | Stemmed | Lemmatised |
|---|---|---|
| terrible | **terribl** | terrible |
| beautiful | **beauti** | beautiful |
| honestly | **honestli** | honestly |
| experience | **experi** | experience |
| dedicated | **dedic** | dedicated |
| employees | employe | **employee** |
| children | children | **child** |

**Why lemmatisation was chosen over stemming:**

- **Stemming** uses simple suffix-stripping rules (Porter algorithm) and frequently produces non-words like `"terribl"`, `"beauti"`, and `"honestli"`. These truncated forms are uninterpretable and would be treated as entirely new vocabulary items by a vectoriser.
- **Lemmatisation** uses WordNet's lexical database to find the true dictionary base form. `"employees"` → `"employee"`, `"children"` → `"child"`, `"running"` → `"running"` (correctly kept, since it is the present participle form). The output is always a real English word.

**Trade-off:** Lemmatisation is significantly slower than stemming because it performs dictionary lookups rather than applying fixed rules. At 271,897 training samples, this is a meaningful computational cost. However, for sentiment classification — where word meaning is central — the quality gain justifies the extra processing time.

---

### 2.8 Data Preparation Output

After preprocessing, the data is split into features and labels:

| Split | Samples |
|---|---|
| `X_train` / `y_train` | 271,897 |
| `X_test` / `y_test` | 69,907 |

The processed text is stored in a new `'processed'` column, preserving the original `'text'` column for reference and auditability. Labels use the original `'rating'` column directly — the integer star ratings that will serve as ground truth for model training and evaluation.